# Igbo ASR tonal evaluation, corrected re-analysis (analysis-2)

Offline re-analysis of `data/metadata.csv` (21 clips, one speaker, `facebook/omniASR-CTC-1B`). No GPU or model needed. It supersedes the metric cells of `analysis.ipynb`, which is kept as the record of the original inference run.

What changed and why (details in `CHANGELOG.md`):
1. D (dropped) and H (added) are computed **per clip**, not netted over the pool.
2. Net loss, drop rate, added rate and DER are reported side by side. The old README mixed net loss and drop rate in one column.
3. `character_error_rate` in the CSV is `1 - difflib ratio`, not CER. True Levenshtein CER is computed here.
4. Bootstrap resamples **clips**, reports undefined resamples, and category contrasts use a paired bootstrap instead of "do the intervals overlap".
5. A position-aware estimate (alignment on base letters) separates correct, wrong, missing and inserted marks.

In [1]:
import sys, pathlib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

root = pathlib.Path.cwd()
if not (root / "src" / "evaluate.py").exists():
    root = root.parent          # running from a subfolder
sys.path.insert(0, str(root))
from src.evaluate import (add_clip_columns, calculate_der, summarize, bootstrap_ci, paired_diff_ci,
                          stat_drop_rate, stat_added_rate, stat_net_loss, stat_der,
                          align_marks, aligned_table, count_tone_marks)
pd.set_option("display.width", 200)

raw = pd.read_csv(root / "data" / "metadata.csv")
df = add_clip_columns(raw)
print(len(df), "clips;", df.category.nunique(), "categories")

21 clips; 4 categories


## 1. Do the counts reproduce the dataset?

Marks are counted as NFD letter clusters carrying grave, acute, macron or dot-below. The recomputed counts should equal the columns in the CSV.

In [2]:
ok_e = (df.diacritics_expected == raw.diacritics_expected).all()
ok_p = (df.diacritics_produced == raw.diacritics_produced).all()
print("expected counts match CSV:", ok_e, "| produced counts match CSV:", ok_p)

expected counts match CSV: True | produced counts match CSV: True


## 2. Metrics, per category

D and H are per clip. Net loss can hide drops when other clips add marks.

In [3]:
t = summarize(df)
cols = ["n_clips","expected","produced","dropped","added","net_loss","drop_rate","added_rate","DER","mean_cer"]
t[cols].round(3)

                      n_clips  expected  produced  dropped  added  net_loss  drop_rate  added_rate    DER  mean_cer
code_switching            5.0      14.0      12.0      7.0    5.0     0.143      0.500       0.357  0.857     0.257
cultural_context          5.0      16.0      15.0      5.0    4.0     0.062      0.312       0.250  0.562     0.374
script_hallucination      5.0      18.0      25.0      2.0    9.0    -0.389      0.111       0.500  0.611     0.285
tonal_diacritics          6.0      49.0      19.0     37.0    7.0     0.612      0.755       0.143  0.898     0.516
overall                  21.0      97.0      71.0     51.0   25.0     0.268      0.526       0.258  0.784     0.365

The old README column "Diacritic Loss" was net loss for three categories and drop rate for the tonal row (61.2% vs 75.5%). Both are shown above.

## 3. Bootstrap over clips

Clips are the independent unit; resampling marks would give intervals that are too narrow. A resample with zero expected marks has an undefined rate and is dropped (count reported).

In [4]:
rows = []
for name, g in list(df.groupby("category")) + [("overall", df)]:
    for label, fn in [("drop_rate", stat_drop_rate), ("added_rate", stat_added_rate), ("DER", stat_der)]:
        r = bootstrap_ci(g, fn, n_boot=10000, seed=42)
        rows.append(dict(group=name, metric=label, point=r["point"], lo=r["lo"], hi=r["hi"], undefined=r["n_undefined"]))
ci = pd.DataFrame(rows)
ci.pivot(index="group", columns="metric", values=["point","lo","hi"]).round(3)

                      point                          lo                          hi                     
metric                  DER added_rate drop_rate    DER added_rate drop_rate    DER added_rate drop_rate
group                                                                                                   
code_switching        0.857      0.357     0.500  0.583      0.000     0.000  1.250      1.000     0.917
cultural_context      0.562      0.250     0.312  0.263      0.059     0.000  1.000      0.571     0.833
overall               0.784      0.258     0.526  0.583      0.090     0.303  1.063      0.587     0.697
script_hallucination  0.611      0.500     0.111  0.238      0.080     0.000  1.889      1.889     0.273
tonal_diacritics      0.898      0.143     0.755  0.640      0.000     0.571  1.556      0.808     0.897

The headline is the **point estimate**; the interval says how much it could move if different clips had been drawn. The bootstrap mean is a diagnostic only.

## 4. Is the tonal category really different? (paired bootstrap)

"The intervals do not overlap" is not a test, and in this data they do overlap for two of three contrasts.

In [5]:
out = []
for other in ["script_hallucination", "cultural_context", "code_switching"]:
    r = paired_diff_ci(df, "category", "tonal_diacritics", other, stat_fn=stat_drop_rate)
    out.append(dict(contrast=f"tonal - {other}", diff_pts=100*r["diff"], lo=100*r["lo"], hi=100*r["hi"],
                    excludes_zero=(r["lo"] > 0 or r["hi"] < 0), undefined=r["n_undefined"]))
pd.DataFrame(out).round(1)

                       contrast  diff_pts    lo    hi  excludes_zero  undefined
0  tonal - script_hallucination      64.4  40.5  85.7           True          2
1      tonal - cultural_context      44.3  -9.4  80.6          False          0
2        tonal - code_switching      25.5 -18.9  78.6          False          3

## 5. Tone marks only

Dot-below letters (ọ, ị, ụ) are partly an orthography choice, and some references omit them. Counting only grave, acute and macron avoids calling a correct ọ an "added" mark.

In [6]:
df["tone_expected"] = df.ground_truth.apply(count_tone_marks)
df["tone_produced"] = df.model_output.apply(count_tone_marks)
tt = df.groupby("category")[["tone_expected","tone_produced"]].sum()
tt.loc["overall"] = tt.sum()
tt

                      tone_expected  tone_produced
category                                          
code_switching                    0              0
cultural_context                  2              0
script_hallucination              0              4
tonal_diacritics                 43              5
overall                          45              9

## 6. Position-aware estimate

Align reference and output on base letters, then classify each mark. Invariants: correct + wrong + missing = expected; correct + wrong + inserted = produced. It is an estimate: a mis-heard letter can make a mark look missing.

In [7]:
at = aligned_table(df)
chk = at.copy()
chk["expected"] = chk.correct + chk.wrong + chk.missing
chk["produced"] = chk.correct + chk.wrong + chk.inserted
exp_prod = df.groupby("category")[["diacritics_expected","diacritics_produced"]].sum()
exp_prod.loc["overall"] = exp_prod.sum()
assert (chk.expected.values == exp_prod.diacritics_expected.values).all()
assert (chk.produced.values == exp_prod.diacritics_produced.values).all()
print("invariants hold")
at

invariants hold


                      correct  wrong  missing  inserted
category                                               
code_switching              7      0        7         5
cultural_context            8      0        8         7
script_hallucination       14      0        4        11
tonal_diacritics            4      7       38         8
overall                    33      7       57        31

## 7. CER versus mark loss

True CER (Levenshtein) against per-clip drop rate, clips with at least one expected mark.

In [8]:
m = df[df.diacritics_expected > 0].copy()
m["drop_rate"] = m.dropped / m.diacritics_expected
print("clips:", len(m))
print("corr(true CER, drop rate)        = %.2f" % np.corrcoef(m.cer, m.drop_rate)[0,1])
print("corr(legacy difflib score, drop) = %.2f" % np.corrcoef(m.seq_dissimilarity, m.drop_rate)[0,1])
print("mean CER: true %.3f | legacy column %.3f" % (df.cer.mean(), raw.character_error_rate.mean()))

clips: 18
corr(true CER, drop rate)        = 0.36
corr(legacy difflib score, drop) = 0.26
mean CER: true 0.365 | legacy column 0.333


## 8. Figures

In [9]:
from src.visualize import plot_loss_by_category, plot_cer_vs_loss, plot_bootstrap_ci
plot_loss_by_category(df)
plot_cer_vs_loss(df)
plot_bootstrap_ci({n: bootstrap_ci(g, stat_drop_rate) for n, g in [("overall", df)] + list(df.groupby("category"))})

## 9. What this does and does not show

- Tonal clips lose most of their marks: 37 of 49 dropped, drop rate 75.5% (CI 57% to 90%), and tone-bearing letters fall from 43 to 5.
- Net loss (61.2% for tonal, 26.8% overall) understates disagreement because 25 marks are added elsewhere; DER is 89.8% and 78.4%.
- Only the contrast with the cross-lingual category excludes zero. The data do not show tonal beating the language-boundary or domain categories.
- Everything rests on 21 clips, one speaker, one annotator, one model. "Consistent with text priors" is a hypothesis for the next experiment (pitch-manipulated controls, second annotator), not a finding.